<a href="https://colab.research.google.com/github/yuri-maradini/TempSal/blob/main/src/train_ueyes_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning TempSAL su UEyes — Colab

Notebook pronto per lanciare il training vero (Step 4) su GPU, invece che sulla CPU locale.

**Prima di eseguire questo notebook**, su Google Drive crea una cartella (default atteso: `MyDrive/TempSAL_UEyes/`) contenente:
- `multilevel_tempsal.pt` — il checkpoint pre-addestrato originale
- `data_ueyes.zip` — l'archivio di `data_ueyes/` generato in locale (Step 1-2)

Poi: **Runtime → Cambia tipo di runtime → GPU**, prima di eseguire le celle.

In [13]:
# Controllo che sia stata assegnata una GPU
!nvidia-smi

Fri Sep  4 15:03:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import os

# Cambia questo path se hai usato un nome/percorso diverso su Drive
DRIVE_DIR = '/content/drive/MyDrive/TempSAL_UEyes'

assert os.path.isdir(DRIVE_DIR), (
    f"Cartella non trovata: {DRIVE_DIR}\n"
    "Creala su Drive e caricaci multilevel_tempsal.pt + data_ueyes.zip prima di continuare."
)
print('Contenuto trovato su Drive:', os.listdir(DRIVE_DIR))

Contenuto trovato su Drive: ['multilevel_tempsal.pt', 'data_ueyes.zip', 'multilevel_tempsal_ueyes.pt', 'multilevel_tempsal_ueyes_v2.pt']


In [16]:
# Codice: sempre aggiornato da GitHub, non serve preparazione
!git clone https://github.com/yuri-maradini/TempSal.git /content/TempSal

fatal: destination path '/content/TempSal' already exists and is not an empty directory.


In [17]:
import shutil

os.makedirs('/content/TempSal/src/checkpoints', exist_ok=True)
shutil.copy(
    f'{DRIVE_DIR}/multilevel_tempsal.pt',
    '/content/TempSal/src/checkpoints/multilevel_tempsal.pt',
)
# multilevel_tempsal_ueyes_v2.pt e' il checkpoint della seconda run (20 epoche,
# epoca 18): questa run riparte da li' -- la testa temporale e' gia' ben
# adattata, l'esperimento isola l'effetto di sbloccare anche il backbone.
shutil.copy(
    f'{DRIVE_DIR}/multilevel_tempsal_ueyes_v2.pt',
    '/content/TempSal/src/checkpoints/multilevel_tempsal_ueyes_v2.pt',
)
print('Checkpoint copiati (originale + v2, il warm-start di questa run).')

Checkpoint copiati (originale + v2, il warm-start di questa run).


In [18]:
import time
import zipfile

# Estratto sul disco locale di Colab (veloce), non lasciato sul mount di Drive
# (l'I/O su Drive montato e' molto piu' lento per tanti file piccoli, e qui
# ce ne sono migliaia tra immagini, mappe e volumi temporali).
t0 = time.time()
with zipfile.ZipFile(f'{DRIVE_DIR}/data_ueyes.zip') as zf:
    zf.extractall('/content/TempSal/')
print(f'Dati estratti in {time.time() - t0:.0f}s')

Dati estratti in 16s


In [19]:
# Controllo veloce di integrita': i conteggi devono combaciare con quelli
# verificati in locale (1872 train / 108 val per ciascuna sottocartella)
for sub in ['images', 'maps', 'fixation_maps', 'saliency_volumes_5', 'fixation_volumes_5']:
    for split in ['train', 'val']:
        d = f'/content/TempSal/data_ueyes/{sub}/{split}'
        n = len(os.listdir(d)) if os.path.isdir(d) else 'MANCANTE'
        print(f'{sub:22s} {split:5s} -> {n}')

images                 train -> 1872
images                 val   -> 108
maps                   train -> 1872
maps                   val   -> 108
fixation_maps          train -> 1872
fixation_maps          val   -> 108
saliency_volumes_5     train -> 9360
saliency_volumes_5     val   -> 540
fixation_volumes_5     train -> 9360
fixation_volumes_5     val   -> 540


In [20]:
# Colab ha gia' PyTorch con supporto CUDA preinstallato: installiamo solo le
# altre dipendenze del progetto, senza toccare torch/torchvision/torchaudio
# (forzare i pin usati in locale, pensati per una build CPU, rischierebbe di
# rimpiazzare la build CUDA gia' pronta di Colab con una incompatibile).
!pip install -q wandb pycocotools ftfy einops clip-anytorch kornia regex

In [21]:
# Verifica che l'installazione sopra non abbia rovinato il supporto CUDA di torch
import torch
print('torch', torch.__version__, '| CUDA disponibile:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU non disponibile: controlla Runtime > Cambia tipo di runtime > GPU'

torch 2.11.0+cu128 | CUDA disponibile: True


## wandb (consigliato per questa run)

Il primo run (10 epoche) è stato fatto con `WANDB_MODE=disabled`: nessuna curva salvata, i numeri per epoca sono stati recuperati a mano dall'output della cella di training. Per questa run vale la pena accendere il logging vero, così le curve di CC/KLDIV/NSS/SIM (aggregate e per-slice) restano disponibili per il confronto e per la tesi senza dover rileggere l'output della cella.

Esegui la cella sotto (chiede l'API key, la trovi su wandb.ai/authorize) prima di lanciare il training. Se preferisci comunque saltarlo, aggiungi di nuovo `WANDB_MODE=disabled` (o `=offline` per salvare i log in locale senza account) davanti al comando `python train.py` nella cella di training.</cell id="HNcrR_LSgcXM">


In [22]:
import wandb
wandb.login()

True

## Training (run 3 — sblocco del backbone)

Le prime due run (10 e 20 epoche, solo testa di `pnas_vol` + layer di mixing allenabili, backbone congelato) hanno raggiunto un plateau: sia la mappa aggregata (~epoca 10) sia il ramo temporale (~epoca 16-18) si sono stabilizzati, e la loss di training continuava a scendere mentre le metriche di validazione restavano piatte — segno che il collo di bottiglia ora è la capacità allenabile del modello, non il numero di epoche.

Questa run prova a sbloccare anche il **backbone PNAS interno di `pnas_vol`**, finora sempre congelato:
- `--model_path`/`--model_vol_path` puntano ora a **`multilevel_tempsal_ueyes_v2.pt`** (il checkpoint della run precedente), non al checkpoint originale — la testa temporale è già ben adattata, questa run isola l'effetto di sbloccare anche il backbone invece di ripartire da zero.
- `--train_enc 1` (era `0` nelle run precedenti) sblocca il backbone di `pnas_vol`.
- `--lr 1e-6`, un ordine di grandezza più basso delle run precedenti (`1e-5`): un backbone pre-addestrato con molti più parametri, su un dataset piccolo (1872 immagini), rischia overfitting/distruzione dei pesi pre-addestrati con un learning rate troppo alto.
- `--no_epochs 10`, poche epoche per lo stesso motivo — meglio guardare presto la curva di validazione (ora salvata su wandb) che rischiare di overfittare a lungo.
- `--model_val_path` → `multilevel_tempsal_ueyes_v3.pt`, così `v2` resta intatto come confronto.

**Aggiornamento dopo un primo tentativo andato in `CUDA out of memory`**: con il backbone sbloccato, PyTorch deve conservare le attivazioni di backprop di un intero PNASNet in più (non solo della piccola testa), e la T4 di Colab (~14.56 GiB utilizzabili) è finita OOM a `--batch_size 32` per un margine minimo (mancavano ~68 MB). Corretto con **gradient accumulation**, aggiunta a `train.py` (nuovo argomento `--grad_accum_steps`, default `1` = comportamento invariato per le run precedenti): `--batch_size 16 --grad_accum_steps 2` dimezza la memoria per singolo forward/backward mantenendo lo stesso batch size *effettivo* (32) delle run 1 e 2, per un confronto equo. Verificato in locale su CPU (mini-dataset, `--train_enc 1 --grad_accum_steps 2`): il backbone di `pnas_vol` cambia correttamente (1378/1380 tensori), `pnas_sal` resta congelato (0/1394) come da progetto. Aggiunto anche `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` come margine extra contro la frammentazione.

Da monitorare più attentamente del solito: il gap train/val (rischio di overfitting più alto con molti più parametri sbloccati) e se il backbone sbloccato porta davvero un guadagno oltre il plateau di `v2`, specialmente sul ramo temporale.

In [23]:
%cd /content/TempSal/src
# PYTORCH_CUDA_ALLOC_CONF: riduce la frammentazione dell'allocatore, utile
# ora che la memoria e' molto piu' vicina al limite della T4 (vedi cella
# markdown sopra sull'OOM del primo tentativo).
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train.py \
  --enc_model pnas_boosted_multi \
  --dataset_dir ../data_ueyes/ \
  --model_path ./checkpoints/multilevel_tempsal_ueyes_v2.pt \
  --model_vol_path ./checkpoints/multilevel_tempsal_ueyes_v2.pt \
  --train_model 1 \
  --train_enc 1 \
  --lr 1e-6 \
  --batch_size 16 \
  --grad_accum_steps 2 \
  --no_epochs 10 \
  --model_val_path ./checkpoints/multilevel_tempsal_ueyes_v3.pt

/content/TempSal/src
usage: train.py [-h] [--no_epochs NO_EPOCHS] [--lr LR] [--kldiv KLDIV]
                [--cc CC] [--nss NSS] [--sim SIM] [--nss_emlnet NSS_EMLNET]
                [--nss_norm NSS_NORM] [--l1 L1] [--lr_sched LR_SCHED]
                [--dilation DILATION] [--enc_model ENC_MODEL] [--optim OPTIM]
                [--load_weight LOAD_WEIGHT] [--kldiv_coeff KLDIV_COEFF]
                [--step_size STEP_SIZE] [--cc_coeff CC_COEFF]
                [--sim_coeff SIM_COEFF] [--nss_coeff NSS_COEFF]
                [--nss_emlnet_coeff NSS_EMLNET_COEFF]
                [--nss_norm_coeff NSS_NORM_COEFF] [--l1_coeff L1_COEFF]
                [--vol_loss_coeff VOL_LOSS_COEFF] [--train_enc TRAIN_ENC]
                [--dataset_dir DATASET_DIR] [--batch_size BATCH_SIZE]
                [--log_interval LOG_INTERVAL] [--no_workers NO_WORKERS]
                [--train_model TRAIN_MODEL] [--time_slices TIME_SLICES]
                [--selected_slices SELECTED_SLICES]
                [--r

In [24]:
# Copia il checkpoint fine-tuned su Drive, cosi' sopravvive alla chiusura
# della sessione Colab. Puoi rieseguire questa cella anche a training ancora
# in corso, per avere un backup intermedio.
import shutil

src_ckpt = '/content/TempSal/src/checkpoints/multilevel_tempsal_ueyes_v3.pt'
dst_ckpt = f'{DRIVE_DIR}/multilevel_tempsal_ueyes_v3.pt'
shutil.copy(src_ckpt, dst_ckpt)
print('Copiato su Drive:', dst_ckpt)

FileNotFoundError: [Errno 2] No such file or directory: '/content/TempSal/src/checkpoints/multilevel_tempsal_ueyes_v3.pt'